**Concept Behind Latent Variable Models:**\
If we define a joint distribution over *observed* **and** *latent* variables, the corresponding distribution of the *observed variables alone* is given by marginalizing over the latent variables. Complex marginal distributions over observed variables can be expressed with *more tractable* joint distributions over the expanded space of observed and latent variables.

## 15.1 K-means Clustering

We can express the error function in k-means as:
$$ J = \sum_{n=1}^N\sum_{k=1}^K r_{nk} \|x_n - \mu_k \|^2 $$
Where $r_{nk}$ is an indicator function, so 1 when the observation is assigned to cluster $k$ and 0 otherwise, and $\mu_k$ is the centroid of cluster $k$.

The well known algorithm to minimize $J$ has two steps:
1. Minimize $J$ w.r.t. $r_{nk}$, keeping $\mu_k$ fixed
2. Minimize $J$ w.r.t. $\mu_{k}$, keeping $r_{nk}$ fixed

The first simply entails assigning each observation to the cluster with the nearest centroid:
$$
r_{nk} = \begin{cases} 1, \ \text{if } \ k = \argmin_i \|x_n - \mu_j\|^2 \\ 0, \ \text{otherwise} \end{cases}
$$
The second amounts to simply recomputing the centroid of each cluster after the reassignment, but it is equivalent to minimizing the error function w.r.t. the centroids.
$$
\mu_k = \frac{\sum_n r_{nk} x_n}{\sum_n r_{nk}} = \frac{\sum_n x_n}{n}
$$

The two steps of this algorithm actually correspond to the two steps of the ***Expectation Maximization*** (EM) algorithm discussed later. Step 1 corresponds to E (expectation) and step 2 corresponds to M (maximization).

## 15.2 Mixtures of Gaussians

Gaussian mixture distributions are linear combinations of Gaussians, with coefficients that are weights:
$$
p(x) = \sum_{k=1}^K \pi_k \mathcal{N}(x|\mu_k, \Sigma_k) \\ \ \\
\pi_k : \ 0 \le \pi_k \le 1, \ \sum_k \pi_k = 1
$$


Let $z$ be a *binary* random vector with $\text{dim}(z) = K$. I.e., $z$ is a one-hot-encoded vector with a $1$-of-$K$ representation. So, $z$ has $K$ possible states according to whichever of its elements are non-zero. Furthermore, let the marginal distribution $p(z)$ by a multinomial distribution of $K$ states specified by the mixing coefficients $\pi_k$, such that: $$p(z_k = 1) = \pi_k$$

Then, the joint distribution of $x$ and $z$ is $p(x, z) = p(x|z) p(z)$, and the conditional distribution of $x$ given $z$ is:
$$
\begin{align*}
& p(x|z_k=1) = \mathcal{N}(x|\mu_k, \Sigma_k) \\
& p(x|z) = \prod_{k=1}^K \mathcal{N}(x|\mu_k, \Sigma_k)^{z_k}
\end{align*}
$$

Then, the *marginal distribution* of $x$ can be obtained by summing the joint distribution over (marginalizing over) all possible states of $z$:
$$p(x) = \sum_z p(x|z)p(z) = \sum_{k=1}^K \pi_k \mathcal{N}(x | \mu_k, \Sigma_k)$$

So, the Gaussian mixture distribution *is* a marginal over the random vector $z$. Or rather, we can equivalently represent the Gaussian mixture using latent variables. This allows us to work with the joint distribution $p(x, z)$ *instead of* the marginal distribution $p(x)$ directly.

Conditioning the other way (via Bayes'):
$$
p(z_k = 1 | x) = \frac{\pi_k \mathcal{N}(x|\mu_k, \Sigma_k)}{\sum_{j=1}^K \pi_j \mathcal{N}(x | \mu_j, \Sigma_j) }
= \frac{p(x|z_k = 1)p(z_k = 1)}{p(x)}
$$

So, $\pi_k$ may be viewed as the **prior** probability of $z_k = 1$, $p(z_k=1 | x)$ is the **posterior** probability once we've observed $x$, and $p(x)$ is the **evidence**.

Solving for $\mu_k$ by setting the partial derivative of the log-likelihood function $\ln p(X|\pi, \mu, \Sigma)$ equal to zero:
$$
\frac{\partial}{\partial \mu} \ln p(X|\pi, \mu, \Sigma) = 0 = \sum_n p(z_{nk} = 1 | x_n) \Sigma^{-1}_k (x_n - \mu_k) \\ \ \\
\mu_k = \frac{1}{N_k} \sum_n p(z_{nk} = 1 | x_n) x_n
$$
Where $N_k$ is the number of observations assigned to cluster $k$.\
Written out: the maximum likelihood estimate for $\mu_k$ for the $k^{th}$ Gaussian component is the weighted mean of *all* observations with weights for each observation $x_n$ given by the posterior probability $p(z_{nk} =1 |x_n)$ that component $k$ was responsible for generating $x_n$.

Solving for $\Sigma_k$ in the same way (i.e. setting derivative w.r.t. $\Sigma_k$ equal to zero) yields:
$$
\Sigma_k = \frac{1}{N_k}\sum_n p(z_{nk} = 1| x_n) (x_n - \mu_k) (x_n-\mu_k)^\intercal
$$
Which has the form of a single Gaussian fit to the data set but with each observations weighted its  posterior probability of being generated by the $k^{th}$ Gaussian component.

Finally, we do the same for $\pi_k$, subject to the constraints that $\pi_k \ge 0$ and $\sum_k \pi_k = 1$. We use a Lagrangian multiplier $\lambda$ to impose this constraint:
$$\argmax_\pi \ \bigg[ p(X | \pi, \mu, \Sigma) + \lambda \big(\sum_k \pi_k - 1\big)\bigg]$$
Which gives:
$$ \pi_k = \frac{N_k}{N} = \frac{1}{N} \sum_{n=1}^N p(z_{nk} = 1| x_n)$$
Which is of course the proportion of observations that are generated by component $k$. Which may be interpreted as the average *responsibility* that component $k$ takes for generating an observation.

## 15.3 Expectation-Maximization (EM) Algorithm

**Basic Premise of EM Algorithm**\
The algorithm has two parts: E and M. In its application to Gaussian mixtures, the first part uses the current parameter values to calculate the posterior probabilities $p(z_{k} = 1 | x)$. Thus, the *expectation* step is a matter of computing the expected posterior. Then, these posterior probabilities are used in the second part to *re-estimate* the means, covariances, and mixing coefficients: $\mu_k, \ \Sigma_k, \ \pi_k$ using the expressions given above. This is the *maximization* step, because it involves maximizing the likelihood of observing $X$ by estimating the parameters (i.e. MLE). **Note** how this two-phase iterative learning approach is *enabled by the latent variable* $z$. 

For observed data $X \in \reals^{N \times D}$, with rows indexing observations $x_n^\intercal$, correspondings latent variable $Z \in \reals^{N \times K}$, with rows $z_n^\intercal$, and set of all parameters $\theta$. The log-likelihood function is: 
$$
\ln p (X|\theta) = \ln\big\{\sum_Z p(X, Z |\theta)\big\} = \sum_{n=1}^N \ln \sum_{z_n} p(x_n, z_n | \theta)
$$

**Step 1 (E):**\
Compute the posterior probabilities $p(z_{nk} = 1 | x_n)$ for all $n\in N, \ k \in K$; i.e:
$$ p (Z|X, \hat{\theta})$$

**Step 2 (M):**\
Compute $\hat{\theta}'$ using the posterior probabilities:
$$\argmax_{\hat{\theta}'} \ \big\{ \sum_Z p(Z|X, \hat{\theta})\ln p(X, Z | \hat{\theta}')\big\}$$

###  General EM ALgorithm
I'm explicating things here for myself.

We want to maximize the log-likelihood of the observed data subject to our learnable parameters $\theta$:
$$\ln p (X|\theta) $$
We can express the distribution over $X$ with a joint distribution of $X$ and a latent variable $Z$:
$$ \ln p(X|\theta) = \ln \frac{p(X, Z|\theta)}{p(Z|X, \theta)} = \ln p(X, Z |\theta) - \ln p(Z|X, \theta)$$

Note that $p(Z|X, \theta)$ is the posterior distribution from: $$p(Z|X, \theta) = \frac{p(X | Z, \theta)p(Z|\theta)}{p(X|\theta)} = \frac{p(X, Z | \theta)}{p(X|\theta)}$$

We cannot observe $Z$ because it is a latent variable. We only observe $X$. Therefore, we cannot know the distribution of $p(X, Z|\theta)$, but we can compute its posterior under known $X$ and $\theta$ because we get to choose its functional (distributional?) form $p(Z)$. This is crucial. If we can find an expression for maximizing $\ln p(X|\theta)$ in terms of the posterior and *not* the joint distribution, then maximization will be tractable.

The model specifies a prior $p(Z|\theta)$ and a likelihood $p(X|Z,\theta)$, which together define the
joint (complete-data) distribution $p(X,Z|\theta)=p(Z|\theta)\,p(X|Z,\theta)$. The joint can be
evaluated for any pair $(X,Z)$. We lack the realized values of $Z$ because it is a fundamentally *unobservable* latent variable. However, we know its distribution (because we specified it when defining the model).

From the joint we derive the marginal $p(X|\theta)=\sum_Z p(X,Z|\theta)$, which is the likelihood we
want to maximize, and the posterior $p(Z|X,\theta)=p(X,Z|\theta)/p(X|\theta)$.

Expressing these as log-likelihoods:
$$\begin{align*}
& \text{Marginal: } \ \ln p(X|\theta) = \ln \sum_Z p(X, Z|\theta) \\
& \text{Posterior: } \ \ln p(Z|X, \theta) = \ln p(X, Z | \theta) - \ln p(X|\theta)
\end{align*}$$

Maximizing $\ln p(X|\theta)$ directly is hard because the sum over $Z$ sits inside the log. The
complete-data log-likelihood $\ln p(X,Z|\theta)$ is easy to maximize, because the log acts on a
product of factors.

EM averages this expression over a posterior with a fixed $\theta$ denoted $\theta^{\text{old}}$ (Bishop's notation, kinda clunky). This posterior is:
$$p(Z|X, \theta^{\text{old}}) \coloneqq q(Z)$$
We denote it $q(Z)$ here for convenience, but also because it no longer varies over $\theta$ since $\theta^{\text{old}}$ is fixed. Averaging in this way (i.e. multiplying by $q(Z)$ then summing over $Z$) yields:
$$
\ln p (X|\theta) = \sum_z q(Z) \ln p(X, Z | \theta) - \sum_Z q(Z) \ln p(Z|X, \theta)
$$
Since $\sum_Z q(Z) = 1$, and $\ln p(X|\theta)$ doesn't depend on $Z$, we can drop the expression from the left-hand-side of the equation. 

Let's break down these terms. The first is:
$$
Q(\theta, \theta^{\text{old}}) = \sum_z q(Z) \ln p(X, Z | \theta)
$$
It is the expected value of the log joint-distribution taken over the posterior distribution under $\theta^{\text{old}}$. Thus, it is our expectation for the log-likelihood of the joint-distribution under our *current* belief about $Z$.

The second term is:
$$\begin{align*}
& -\sum_Z q(Z) \ln p(Z|X, \theta) \\
& \;=\; -\sum_Z q(Z) \ln \bigg[\frac{q(Z)p(Z|X, \theta)}{q(Z)}\bigg] \\
& \;=\; -\sum_Z q(Z) \ln\bigg[\frac{p(Z|X, \theta)}{q(Z)}\bigg] - \sum_Z q(Z) \ln q(Z) \\
& \;=\; \sum_Z q(Z) \ln\bigg[\frac{q(Z)}{p(Z|X, \theta)}\bigg] - \sum_Z q(Z) \ln q(Z) \\
& \;=\; \text{KL}\big(q(Z) \| p(Z|X, \theta)\big) + H[q(Z)]
\end{align*}$$

So, the full expression is:
$$
\ln p (X|\theta) = Q(\theta, \theta^{\text{old}}) + H[q(Z)] + \text{KL}\big(q(Z) \| p(Z|X, \theta)\big)
$$

The entropy $H[q(Z)]$ *does not depend* on $\theta$. Also, $\text{KL}(\cdot)$ is *always non-negative* and $\text{KL} = 0$ *only* at $\theta = \theta^\text{old}$.\
Therefore,
$$ \ln p(X|\theta) - \ln p(X|\theta^\text{old}) \ge Q(\theta, \theta^\text{old}) - Q(\theta^\text{old},\theta^\text{old}) $$
So, any $\theta$ that increases $Q$ **cannot** decrease the log-likelihood.

So, we select $\theta^\text{new}$ such that:
$$\theta^\text{new} = \argmax_\theta Q(\theta, \theta^\text{old})$$
Which guarantees that:
$$ \ln p(X|\theta^\text{new}) \ge \ln p(X|\theta^\text{old}) $$

**The EM Algorithm:**
1. *Expectation* (E-step)\
Compute $q(Z) = p(Z|X, \theta^\text{old})$ to define $Q(\theta, \theta^\text{old})$
2. *Maximization* (M-step)\
Select $\theta^\text{new} = \argmax_\theta Q(\theta, \theta^\text{old})$

That's it.